# Tutorial: Fragment Embedding Visualization and Clustering

## Learning Objectives

By the end of this tutorial, you will be able to:

1. **Load or generate fragment embeddings** from a trained GAE encoder
2. **Create 2D and 3D UMAP visualizations** of fragment chemical space
3. **Create interactive t-SNE plots** with Plotly
4. **Create PCA plots** with variance-explained axis labels
5. **Cluster fragments** based on learned embeddings
6. **Interpret clustering results** to understand fragment similarity
7. **Visualize cluster examples** to examine chemical patterns
8. **Customize visualizations** for publication or exploratory analysis
9. **Export plots** in various formats (HTML, PNG)

## Table of Contents

1. [Introduction to Fragment Visualization](#1-introduction-to-fragment-visualization)
2. [Setup and Imports](#2-setup-and-imports)
3. [Loading Vocabulary and Model](#3-loading-vocabulary-and-model)
4. [Generating Fragment Embeddings](#4-generating-fragment-embeddings)
5. [Creating Clustering Object](#5-creating-clustering-object)
6. [2D t-SNE Visualization](#6-2d-t-sne-visualization)
7. [2D UMAP Visualization](#7-2d-umap-visualization)
8. [2D PCA Visualization](#8-2d-pca-visualization)
9. [3D Visualizations (t-SNE, UMAP, and PCA)](#9-3d-visualizations)
10. [Comparing t-SNE vs UMAP vs PCA](#10-comparing-t-sne-vs-umap-vs-pca)
11. [Vocab vs Corpus Embedding Comparison](#11-vocab-vs-corpus-embedding-comparison)
12. [Visualizing Cluster Examples](#12-visualizing-cluster-examples)
13. [Analyzing Cluster Quality](#13-analyzing-cluster-quality)
14. [Exporting Visualizations](#14-exporting-visualizations)
15. [Summary and Next Steps](#15-summary-and-next-steps)

---

## 1. Introduction to Fragment Visualization

The **Graph Autoencoder (GAE)** learns to represent molecular fragments as **128-dimensional continuous vectors** (embeddings). These high-dimensional embeddings capture structural and chemical similarity between fragments.

To visualize and interpret these embeddings, we use **dimensionality reduction** techniques:

### t-SNE (t-Distributed Stochastic Neighbor Embedding)
- **Preserves local structure**: Similar fragments cluster together
- **Non-linear**: Can reveal complex relationships
- **Non-deterministic**: Different runs produce different layouts
- **Best for**: Exploratory visualization, finding clusters

### UMAP (Uniform Manifold Approximation and Projection)
- **Preserves both local and global structure**: Better reflects overall topology
- **Faster** than t-SNE for large datasets
- **More consistent**: Similar results across runs (with fixed random_state)
- **Best for**: Publication figures, comparing multiple datasets

### PCA (Principal Component Analysis)
- **Linear**: Preserves global variance structure
- **Deterministic**: Always produces the same result
- **Interpretable axes**: Each component explains a percentage of the total variance
- **Fastest**: No iterative optimization
- **Best for**: Quick overview, understanding dominant variance directions

### Why Visualize Fragment Embeddings?

1. **Validate training**: Well-trained models cluster chemically similar fragments
2. **Discover patterns**: Identify fragment families (aromatics, aliphatics, functional groups)
3. **Guide vocabulary expansion**: Find gaps in chemical space coverage
4. **Interpret downstream models**: Understand which fragments drive predictions
5. **Quality assessment**: Check if embeddings capture meaningful chemical properties

---

## 2. Setup and Imports

We'll use:
- **GSGE**: For loading vocabulary and generating embeddings
- **PyTorch**: For loading trained encoder weights
- **UMAP**: For dimensionality reduction
- **Plotly**: For interactive 3D visualizations
- **RDKit**: For chemical structure rendering
- **Seaborn**: For color palettes

In [1]:
# GSGE and GAE imports
from GSGE import GSGE, get_tests_dir
from GSGE.graphs.fragment_graph.GAE import AttentiveFP

# RDKit for chemistry
from rdkit import Chem

# Numerical and scientific computing
import numpy as np
import pandas as pd

# Dimensionality reduction
import umap

# Visualization
import seaborn as sns
import plotly.express as px


# Utilities
import random
from pathlib import Path

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("[OK] Imports successful")

# Check for CUDA availability
import torch
if torch.cuda.is_available():
    print("[OK] CUDA is available")
else:
    print("[OK] CUDA is not available, using CPU")

[OK] Imports successful
[OK] CUDA is available


---

## 3. Loading Vocabulary and Model

We need:
1. **A trained vocabulary** (GS_vocab)
2. **A trained GAE encoder** (checkpoint file) - OR pre-loaded embeddings

### Option 1: Use Pre-Trained Package Data (Quickest)

GSGE includes pre-trained vocabulary and embeddings for demonstration:

In [2]:
# Load pre-trained GSGE vocabulary from package test data
tests_dir = get_tests_dir()

if tests_dir is None:
    raise RuntimeError(
        "Cannot find tests directory. This notebook requires running from "
        "a source checkout with the tests/ directory available."
    )

pkl_path = tests_dir / 'test_gsge_save_with_descriptors.pkl'

if not pkl_path.exists():
    raise FileNotFoundError(
        f"Test data file not found: {pkl_path}\n"
        f"Please ensure the tests directory has the required test fixtures."
    )

gsge = GSGE(GSGE_load_path=str(pkl_path))

print(f"[OK] Loaded vocabulary with {gsge.get_GS_vocab().num_fragments} fragments")

Data loaded and restored into gsge from C:\Users\bolak\OneDrive - Universiteit Leiden\PhD\Repos\jasper\gsge\tests\test_gsge_save_with_descriptors.pkl
[OK] Loaded vocabulary with 185 fragments


### Option 2: Use Your Own Vocabulary and Checkpoint

If you have a custom vocabulary and trained GAE checkpoint:

In [3]:
# # Example: Load custom vocabulary and corpus
# vocab_dir = '../00_making_vocabs/vocabs'
# vocab_path = os.path.join(vocab_dir, 'GS_vocab_v5')
# corpus_path = os.path.join(vocab_dir, 'GSGE_corpus_v5')

# # Create GSGE instance with both vocabulary and corpus
# gsge = GSGE(GS_vocab=vocab_path, GSGE_corpus=corpus_path)

# # Define encoder architecture (must match training)
# encoder = AttentiveFP(
#     in_channels=9, 
#     hidden_channels=256, 
#     out_channels=128,
#     edge_dim=3, 
#     num_layers=3, 
#     num_timesteps=2
# )

# # Set encoder in GSGE
# gsge.set_encoder(encoder)

# # Load checkpoint
# checkpoint_path = 'model_checkpoints/checkpoint_epoch_300.pth'
# gsge.load_GAE_weights(checkpoint_path)

# print("[OK] Custom vocabulary and model loaded")
# print(f"  - Vocabulary fragments: {gsge.get_GS_vocab().num_fragments}")
# print(f"  - Corpus fragments: {gsge.get_GSGE_corpus().num_fragments}")

### Understanding Vocabulary vs. Corpus:

- **GS_Vocab**: Merged, generalized fragments (~100-500 fragments)
  - Used for compound graphs and tokenization
  - Fragments are generalized (R-groups collapsed)
  - **This is what we'll visualize**
  
- **GSGE_Corpus**: Non-merged fragments (~1000-10000 fragments)
  - Used for GAE training
  - Contains more chemical diversity
  - Preserves specific R-groups

---

## 4. Generating Fragment Embeddings

If you loaded a complete GSGE object (Option 1), embeddings are already available. If you're using Option 2, you need to generate them using a trained encoder.

### For Pre-Trained Package Data:

In [4]:
# Check if embeddings are available
if gsge.get_fragment_embeddings() is not None:
    print("[OK] Embeddings already loaded from saved GSGE object")
    
    # Extract embeddings as numpy array
    fragment_smiles = gsge.get_fragments_smiles()
    embeddings = gsge.get_fragment_embeddings()
    
    # Filter out invalid SMILES (required for MCS clustering)
    # Note: Fragment SMILES use '*1' for attachment points, which conflicts
    # with RDKit's ring closure syntax. Replace '*1' with '*' before parsing.
    valid_indices = []
    valid_smiles = []
    for i, smi in enumerate(fragment_smiles):
        clean_smi = smi.replace('*1', '*')
        mol = Chem.MolFromSmiles(clean_smi)
        if mol is not None:
            valid_indices.append(i)
            valid_smiles.append(smi)
    
    if len(valid_indices) < len(fragment_smiles):
        print(f"  [!] Filtered out {len(fragment_smiles) - len(valid_indices)} invalid SMILES")
    
    # Create smiles_df for clustering with only valid SMILES
    smiles_df = pd.DataFrame({'SMILES': valid_smiles})
    
    # Filter embeddings to match valid SMILES
    embeddings = embeddings[valid_indices]
    
    graph_data = None  # Not needed when embeddings and smiles_df are provided
    
    print(f"  Embedding shape: {embeddings.shape}")
    print(f"  Valid fragments: {len(valid_smiles)}")
else:
    print("[!] No embeddings found. You need to train a GAE and generate embeddings first.")
    print("  See: use_examples/03_GAE/train_GAE.py")
    smiles_df = None
    embeddings = None
    graph_data = None

[OK] Embeddings already loaded from saved GSGE object
  Embedding shape: torch.Size([185, 128])
  Valid fragments: 185


### For Custom Vocabulary (if using Option 2):

In [5]:
# # If embeddings are not available, generate them:
# device = 'cuda' if torch.cuda.is_available() else 'cpu'

# print(f"Using device: {device}")

# # Generate embeddings for vocabulary fragments
# print("\nGenerating embeddings for vocabulary fragments...")
# embeddings, graph_data = gsge.embed_fragments(
#     frag_smiles=gsge.get_fragments_smiles(),
#     load_checkpoint_path=checkpoint_path,
#     device=device,
#     return_data=True
# )
# print(f"  - Vocabulary embeddings shape: {embeddings.shape}")

### Understanding the Embeddings:

Each fragment is represented as a **128-dimensional vector**:
- Similar fragments have similar embeddings (close in embedding space)
- Dissimilar fragments have different embeddings (far apart)
- The embeddings capture chemical and structural properties learned during GAE training

---

## 5. Creating Clustering Object

The `GSGE_clustering` class provides convenient methods for visualization and analysis:

In [6]:
# Create clustering object
gsge_clustering = gsge.get_GSGE_clustering(
    embeddings=embeddings,
    smiles_df=smiles_df,  # Required for clustering when using pre-computed embeddings
    graph_data=graph_data  # Can be None when smiles_df is provided
)

print(f"[OK] Clustering object created")
print(f"  Number of fragments: {len(gsge_clustering.smiles_df)}")
print(f"  Embedding dimension: {gsge_clustering.embeddings.shape[1]}")
print(f"  Number of clusters: {len(np.unique(gsge_clustering.cluster_labels))}")
if hasattr(gsge_clustering, 'cophenetic_coefficient'):
    print(f"  Cophenetic coefficient: {gsge_clustering.cophenetic_coefficient:.4f}")
    print(f"\nCophenetic coefficient interpretation:")
    print(f"  >0.8: Excellent clustering structure")
    print(f"  0.6-0.8: Good clustering")
    print(f"  <0.6: Weak clustering (may need more training or better hyperparameters)")

Chunk Size: 10000
Number of Workers: 20
Number of unique SMILES for clustering: 185


Calculating similarities in chunks:   0%|          | 0/2 [00:00<?, ?chunk/s]

Cophenetic coefficient calculated: 0.9535
[OK] Clustering object created
  Number of fragments: 185
  Embedding dimension: 128
  Number of clusters: 6


### Interpreting Clusters:

Clusters typically correspond to:
- **Functional group families**: Aromatics, amines, carbonyls, etc.
- **Structural motifs**: Ring systems, linear chains, branched structures
- **Chemical properties**: Polarity, reactivity, size

---

## 6. 2D t-SNE Visualization

The built-in `plot_2D_TSNE()` method creates an interactive Plotly scatter plot where:
- **Each point** is a molecular fragment
- **Colors** represent automatically discovered clusters
- **Hover** shows SMILES structures
- **Zoom/pan** with mouse controls

**Note**: t-SNE is non-deterministic, so the layout changes each run. For reproducible plots, use UMAP (next sections).

In [ ]:
# Create 2D t-SNE plot
fig_tsne_2d = gsge_clustering.plot_2D_TSNE(
    random_state=42,
    scatter_2d_args={'height': 700, 'width': 1000}
)

# Customize appearance
fig_tsne_2d.update_layout(
    title="2D t-SNE Visualization of Fragment Embeddings",
    font=dict(size=14)
)

fig_tsne_2d.show()

print("\n[TIP] Tip: Hover over points to see SMILES, use toolbar to zoom and pan")

---

## 7. 2D UMAP Visualization

UMAP provides a more stable alternative to t-SNE. 2D projections are useful for:
- **Publication figures**: Easier to interpret than 3D
- **Comparing multiple datasets**: Consistent layout
- **Print media**: No rotation needed

In [ ]:
# Create 2D UMAP plot using built-in method
fig_umap_2d = gsge_clustering.plot_2D_UMAP(
    random_state=42,
    scatter_2d_args={'height': 700, 'width': 1000}
)

# Customize appearance
fig_umap_2d.update_layout(
    title="2D UMAP Visualization of Fragment Embeddings",
    font=dict(size=14)
)

fig_umap_2d.show()

print("\n[OK] 2D UMAP complete")
print("[TIP] Tip: This layout is deterministic - it will look the same every time with random_state=42")

---

## 8. 2D PCA Visualization

PCA provides a linear projection that is **deterministic** and **fast**. Unlike t-SNE and UMAP, the axes are interpretable — each principal component explains a percentage of the total variance in the embeddings. This makes PCA a good first-look method and useful for checking how much structure is captured by just two linear dimensions.

In [ ]:
# Create 2D PCA plot
fig_pca_2d = gsge_clustering.plot_2D_PCA(
    scatter_2d_args={'height': 700, 'width': 1000}
)

# Customize appearance
fig_pca_2d.update_layout(
    title="2D PCA Visualization of Fragment Embeddings",
    font=dict(size=14)
)

fig_pca_2d.show()

print("\n[OK] 2D PCA complete")
print("[TIP] Tip: Axis labels show the % of variance explained by each component")

---

## 9. 3D Visualizations

3D projections provide more visual separation between clusters while maintaining reproducibility (for UMAP and PCA).

### 9.1 3D t-SNE Visualization

In [ ]:
# Create 3D t-SNE plot
fig_tsne_3d = gsge_clustering.plot_3D_TSNE()

# Customize appearance
fig_tsne_3d.update_layout(
    title="3D t-SNE Visualization of Fragment Embeddings",
    height=800,
    width=1200,
    font=dict(size=14)
)

fig_tsne_3d.show()

print("\n[TIP] Tip: Click and drag to rotate, scroll to zoom, hover over points to see SMILES")

### 9.2 3D UMAP Visualization

In [ ]:
# Create 3D UMAP plot using built-in method
fig_umap_3d = gsge_clustering.plot_3D_UMAP(
    random_state=42,
    scatter_3d_args={'height': 800, 'width': 1400}
)

# Improve styling
fig_umap_3d.update_traces(marker=dict(size=5, line=dict(width=0.3, color='DarkSlateGrey')))
fig_umap_3d.update_layout(
    font=dict(size=14),
    scene=dict(
        xaxis_title="UMAP-1",
        yaxis_title="UMAP-2",
        zaxis_title="UMAP-3"
    )
)

fig_umap_3d.show()

print("\n[OK] 3D UMAP complete")
print("[TIP] Tip: Drag to rotate, scroll to zoom, double-click to reset view")

### 9.3 3D PCA Visualization

In [ ]:
# Create 3D PCA plot
fig_pca_3d = gsge_clustering.plot_3D_PCA(
    scatter_3d_args={'height': 800, 'width': 1400}
)

fig_pca_3d.update_traces(marker=dict(size=5, line=dict(width=0.3, color='DarkSlateGrey')))
fig_pca_3d.update_layout(font=dict(size=14))

fig_pca_3d.show()

print("\n[OK] 3D PCA complete")
print("[TIP] Tip: Axis labels show the % of variance explained by each principal component")

---

## 10. Comparing t-SNE vs UMAP vs PCA

Let's compare the three approaches:

| Feature | t-SNE | UMAP | PCA |
|---------|-------|------|-----|
| **Type** | Non-linear | Non-linear | Linear |
| **Local structure** | Excellent | Good | Moderate |
| **Global structure** | Poor | Excellent | Good (variance-based) |
| **Reproducibility** | Low | High (with random_state) | Deterministic |
| **Speed** | Slowest | Fast | Fastest |
| **Interpretable axes** | No | No | Yes (% variance) |
| **Best for** | Exploratory clustering | Publication figures | Quick overview, variance analysis |

### Practical Recommendations:

1. **Use PCA for**:
   - Quick first look at embedding structure
   - Understanding how much variance is captured by top components
   - When you need fully deterministic, reproducible results
   - Supplementary material showing linear separability

2. **Use t-SNE for**:
   - Initial exploration of clusters
   - Finding unexpected groupings
   - When you don't need reproducibility

3. **Use UMAP for**:
   - Publication figures
   - Comparing multiple vocabularies
   - Understanding overall chemical space topology
   - When you need consistent results

4. **Use all three**:
   - Validate that clustering is consistent across methods
   - PCA tells you about variance, t-SNE/UMAP about manifold structure

5. **3D vs 2D**:
   - **2D**: Better for publications, presentations, print
   - **3D**: Better for interactive exploration, finding separation

---

## 11. Vocab vs Corpus Embedding Comparison

The **vocabulary** (GS_Vocab) contains merged, diverse fragments used for molecule representation, while the **corpus** (GSGE_Corpus) contains non-merged fragments used for GAE training. By projecting both into the same 2D space we can see how well the vocabulary covers the corpus distribution and whether the GAE learned a smooth latent space.

Corpus fragments are shown as **light grey background points**; vocabulary fragments are coloured by cluster on top.

In [ ]:
# --- Vocab vs Corpus overlay ---

corpus_obj = gsge.get_GSGE_corpus()

if corpus_obj is None:
    print("[!] No corpus loaded in this GSGE object -- skipping corpus overlay.")
    fig_tsne_corpus = None
    fig_umap_corpus = None
    fig_pca_corpus = None
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    corpus_frags = corpus_obj.fragments

    # Filter to valid SMILES
    corpus_valid = [s for s in corpus_frags if Chem.MolFromSmiles(s) is not None]
    print(f"Corpus fragments: {len(corpus_frags)} total, {len(corpus_valid)} valid SMILES")

    # Remove fragments already in vocab to avoid duplicates
    vocab_smiles_list = gsge_clustering.smiles_df["SMILES"].tolist()
    vocab_set = set(vocab_smiles_list)
    corpus_only = [s for s in corpus_valid if s not in vocab_set]
    print(f"Corpus-only fragments (not in vocab): {len(corpus_only)}")

    # Embed corpus fragments
    print("Embedding corpus fragments...")
    corpus_embeddings, _ = gsge.embed_fragments(
        frag_smiles=corpus_only,
        device=device,
        return_data=True,
    )
    print(f"Corpus embedding shape: {corpus_embeddings.shape}")

    # t-SNE overlay
    print("Computing t-SNE on combined vocab + corpus...")
    fig_tsne_corpus = gsge_clustering.plot_2D_TSNE_with_corpus(
        corpus_embeddings=corpus_embeddings,
        corpus_smiles=corpus_only,
        random_state=42,
        scatter_2d_args={"height": 700, "width": 1000},
    )
    fig_tsne_corpus.show()

    # UMAP overlay
    print("Computing UMAP on combined vocab + corpus...")
    fig_umap_corpus = gsge_clustering.plot_2D_UMAP_with_corpus(
        corpus_embeddings=corpus_embeddings,
        corpus_smiles=corpus_only,
        random_state=42,
        scatter_2d_args={"height": 700, "width": 1000},
    )
    fig_umap_corpus.show()

    # PCA overlay
    print("Computing PCA on combined vocab + corpus...")
    fig_pca_corpus = gsge_clustering.plot_2D_PCA_with_corpus(
        corpus_embeddings=corpus_embeddings,
        corpus_smiles=corpus_only,
        scatter_2d_args={"height": 700, "width": 1000},
    )
    fig_pca_corpus.show()

    print("[OK] Corpus overlay plots complete")

---

## 11. Visualizing Cluster Examples

To better understand what fragments are in each cluster, we can visualize example molecules from each cluster:

In [ ]:
# Import plot_cluster_grid from GSGE.visualization
from GSGE.visualization import plot_cluster_grid

print("[OK] plot_cluster_grid imported from GSGE.visualization")

In [ ]:
# Get SMILES and labels from clustering object
vocab_smiles = gsge_clustering.smiles_df['SMILES'].tolist()
cluster_labels = gsge_clustering.cluster_labels

# Create color map for clusters
unique_clusters = np.unique(cluster_labels)
palette = sns.color_palette("hsv", len(unique_clusters))
cluster_color_map = {label: palette[i] for i, label in enumerate(unique_clusters)}

# Plot cluster examples
print("Generating cluster examples...")
plot_cluster_grid(
    vocab_smiles,
    cluster_labels,
    cluster_color_map,
    samples_per_cluster=5
)

print("[OK] Cluster examples displayed")

---

## 12. Analyzing Cluster Quality

Good embeddings should show:
- **Clear cluster separation**: Different fragment types group together
- **Chemically meaningful clusters**: E.g., aromatics vs. aliphatics
- **Smooth transitions**: Gradual changes in chemical properties
- **No outliers**: All fragments fit into some cluster

In [ ]:
# Analyze cluster composition
from collections import Counter

print("=== Cluster Analysis ===")
print(f"\nTotal fragments: {len(gsge_clustering.cluster_labels)}")
print(f"Total clusters: {len(np.unique(gsge_clustering.cluster_labels))}")

# Cluster size distribution
cluster_sizes = Counter(gsge_clustering.cluster_labels)
print(f"\nCluster size distribution:")
for cluster_id, size in sorted(cluster_sizes.items(), key=lambda x: -x[1])[:10]:
    print(f"  Cluster {cluster_id}: {size} fragments")

# Show example fragments from largest cluster
largest_cluster = max(cluster_sizes.keys(), key=lambda k: cluster_sizes[k])
cluster_mask = gsge_clustering.cluster_labels == largest_cluster
cluster_smiles = gsge_clustering.smiles_df[cluster_mask]['SMILES'].tolist()

print(f"\n[INFO] Example fragments from largest cluster (Cluster {largest_cluster}):")
for i, smiles in enumerate(cluster_smiles[:5], 1):
    print(f"  {i}. {smiles}")

print("\n[TIP] Tip: Inspect multiple clusters to understand what chemical patterns the model learned")

### Analyzing Cluster Quality

**Good clustering** should show:
- **Clear separation**: Distinct clusters with minimal overlap
- **Chemical meaning**: Clusters correspond to functional groups or structural motifs
- **Balanced sizes**: No extremely large or tiny clusters (unless justified)
- **Smooth transitions**: Similar clusters should be adjacent in embedding space

**Signs of problems**:
- **No clear structure**: May need more GAE training
- **One giant cluster**: Model may not be learning distinctive features
- **Many single-point clusters**: Overfitting or too many clusters

---

## 14. Exporting Visualizations

Save plots for publications or presentations:

In [ ]:
# Create output directory
output_dir = Path("fragment_visualizations")
output_dir.mkdir(exist_ok=True)

# Collect all figures with their base names
figures = {
    "fragment_tsne_2d": fig_tsne_2d,
    "fragment_tsne_3d": fig_tsne_3d,
    "fragment_umap_2d": fig_umap_2d,
    "fragment_umap_3d": fig_umap_3d,
    "fragment_pca_2d": fig_pca_2d,
    "fragment_pca_3d": fig_pca_3d,
}
if fig_tsne_corpus is not None:
    figures["corpus_overlay_tsne_2d"] = fig_tsne_corpus
if fig_umap_corpus is not None:
    figures["corpus_overlay_umap_2d"] = fig_umap_corpus
if fig_pca_corpus is not None:
    figures["corpus_overlay_pca_2d"] = fig_pca_corpus

# Export to HTML (interactive)
for name, fig in figures.items():
    fig.write_html(output_dir / f"{name}.html")

print(f"[OK] Exported {len(figures)} interactive HTML plots to {output_dir}/")

# Export to PNG (300 DPI), SVG, and PDF (requires kaleido)
# 300 DPI: scale=3 with default 100 DPI base ~ 300 DPI effective resolution
try:
    for name, fig in figures.items():
        w = 1400 if "3d" in name else 1000
        h = 800 if "3d" in name else 700
        fig.write_image(output_dir / f"{name}.png", width=w, height=h, scale=3)
        fig.write_image(output_dir / f"{name}.svg", width=w, height=h)
        fig.write_image(output_dir / f"{name}.pdf", width=w, height=h)
    print(f"[OK] Exported PNG (300 DPI), SVG, and PDF for {len(figures)} figures to {output_dir}/")
except Exception as e:
    print(f"\n[!] Could not export static images (install kaleido: pip install kaleido)")
    print(f"  Error: {e}")

---

## 15. Summary and Next Steps

### What We Learned

1. [OK] **Loaded/generated fragment embeddings** from trained GAE encoder
2. [OK] **Created interactive visualizations** using t-SNE, UMAP, and PCA (2D and 3D)
3. [OK] **Compared 2D vs 3D projections** for different use cases
4. [OK] **Analyzed cluster quality** to validate model training
5. [OK] **Visualized cluster examples** to understand chemical patterns
6. [OK] **Exported visualizations** for publications and presentations
7. [OK] **Understood when to use t-SNE vs UMAP vs PCA**

### Key Takeaways

- **Well-trained GAE models** produce clear, chemically meaningful clusters
- **UMAP** is generally better for reproducible, publication-quality figures
- **t-SNE** is great for exploratory analysis and finding unexpected patterns
- **PCA** is fastest and provides interpretable axes (% variance explained)
- **3D visualizations** provide more separation but are harder to print
- **Interactive Plotly plots** allow exploring fragment structures by hovering
- **Cluster inspection** helps validate that embeddings capture chemistry

### Visualization Guide

| Goal | Recommended Method | Format |
|------|-------------------|--------|
| **Publication figures** | 2D UMAP | PNG/SVG |
| **Quick overview** | 2D PCA | PNG/SVG |
| **Variance analysis** | 2D/3D PCA | PNG (check axis labels) |
| **Presentations** | 2D UMAP or 3D UMAP | HTML or PNG |
| **Exploratory analysis** | t-SNE (2D or 3D) | HTML (interactive) |
| **Comparing datasets** | 2D UMAP (fixed random_state) | PNG side-by-side |
| **Archive/sharing** | All formats | HTML (interactive) + PNG (static) |

### Next Steps

1. **Analyze specific clusters**:
   - Extract fragments from interesting clusters
   - Visualize fragment structures with RDKit
   - Understand what chemical patterns the model learned

2. **Use embeddings in downstream tasks**:
   ```python
   from GSGE.embedding import GSGE_Embedding
   
   emb_layer = GSGE_Embedding(
       0, None, 0,
       gsge.get_fragment_embeddings(),
       only_token2vec=True,
       no_grad=True
   )
   # Use in PyTorch models for property prediction
   ```

3. **Compare multiple vocabularies**:
   - Create UMAP plots for different training runs
   - Assess how vocabulary changes affect embedding quality

4. **Validate cluster interpretations**:
   - Calculate RDKit descriptors for each cluster
   - Verify that clusters correspond to chemical intuition

5. **Iterate on GAE training**:
   - If clusters are poor, retrain with:
     - More epochs
     - Different hyperparameters
     - Expanded corpus

### Troubleshooting

- **No clear clusters**: Model may need more training or larger capacity
- **Too many small clusters**: Try adjusting clustering threshold in `get_GSGE_clustering()`
- **Outliers**: Some fragments may be unique - consider expanding vocabulary
- **UMAP warnings about n_jobs**: Normal - UMAP sets n_jobs=1 when random_state is fixed
- **Memory issues**: Use smaller perplexity for t-SNE or reduce dataset size

### Additional Resources

- [Tutorial README](README.md) - GAE training overview
- [t-SNE Paper](https://jmlr.org/papers/v9/vandermaaten08a.html) - Understanding t-SNE
- [UMAP Documentation](https://umap-learn.readthedocs.io/) - UMAP guide
- [AttentiveFP Paper](https://pubs.acs.org/doi/10.1021/acs.jmedchem.9b00959) - Model architecture
- [GSGE Documentation](../../docs/index.md) - Complete package docs

---